In [1]:
from train_plant2_from_carl_trajectories import CarlPlantTrajectoryDataset

dataset = CarlPlantTrajectoryDataset("/home/jovyan/shares/SR006.nfs2/arbelyaev/sdc/pdd-bench/data")

/home/jovyan/.mlspace/envs/plant2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
dataset[45]["plant2_batch"]["x_objs"][2]

array([ 4.       , 29.672228 , -1.1237534,  0.       ,  0.       ,
        1.       ,  1.       ], dtype=float32)

In [15]:
import torch

def get_two_hot_encoding(
        target_speed: torch.Tensor,
        config_target_speeds: torch.Tensor,
        brake: torch.Tensor,
    ) -> torch.Tensor:
        """
        Two-hot encoding as used in the original PlanT lit_module.py.
        target_speed:        (B,)  – ego speed in m/s, must be >= 0
        config_target_speeds:(C,)  – sorted bin centres
        brake:               (B,)  – bool, True → force speed=0 bin
        returns:             (B, C) soft label distribution
        """
        if torch.any(target_speed < 0):
            raise ValueError("Target speed value must be non-negative for two-hot encoding.")

        labels = torch.zeros(target_speed.shape[0], len(config_target_speeds), device=target_speed.device)

        diffs = (config_target_speeds > target_speed[:, None]).float()   # (B, C)
        vals, idxs = diffs.max(dim=1)

        upper_ind = idxs
        lower_ind = (idxs - 1)
        upper_val = config_target_speeds[upper_ind]
        lower_val = config_target_speeds[lower_ind]

        denom = (upper_val - lower_val)
        lower_weight = (upper_val - target_speed) / denom
        upper_weight = (target_speed - lower_val) / denom

        labels[torch.arange(target_speed.shape[0]), lower_ind] = lower_weight
        labels[torch.arange(target_speed.shape[0]), upper_ind] = upper_weight

        # Clear rows where brake or no config value greater than target speed
        labels[torch.logical_or(brake, vals==0)] = 0

        # Set brake rows to 0
        labels[brake, 0] = 1.0

        # Set rows with max speed and without brake pressed to last bin
        labels[torch.logical_and(vals==0, ~brake), -1] = 1.0

        return labels

In [6]:
import numpy as np

bins_speeds = torch.tensor(np.array([0, 0.025, 0.05472609, 1.0, 1.5, 2.0, 4.0, 8.0, 10.0, 20.0], dtype=np.float64), dtype=torch.float32)

In [18]:
bins_speeds[1] * 0.4913

tensor(0.0123)

In [16]:
speeds = []
speeds_two_hot = []

for i in range(100):
    input_speeds = torch.tensor(dataset[i]["plant2_batch"]["target_speed"], dtype=torch.float32).squeeze(1)

    two_hot = get_two_hot_encoding(
        input_speeds,
        bins_speeds,
        brake = torch.zeros_like(input_speeds, dtype=torch.bool) 
    )

    speeds.append(input_speeds)
    speeds_two_hot.append(two_hot)

    if input_speeds < 0.1:
        print(input_speeds)
        print(two_hot)
        print("="*100)

speeds = torch.cat(speeds)
speeds_two_hot = torch.cat(speeds_two_hot)


tensor([0.])
tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([0.0757])
tensor([[0.0000, 0.0000, 0.9778, 0.0222, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
tensor([0.0393])
tensor([[0.0000, 0.5196, 0.4804, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
tensor([0.0029])
tensor([[0.8858, 0.1142, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
tensor([0.0080])
tensor([[0.6791, 0.3209, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
tensor([0.0123])
tensor([[0.5087, 0.4913, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
tensor([0.0547])
tensor([[0.0000e+00, 0.0000e+00, 1.0000e+00, 3.9410e-09, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]])
tensor([0.0184])
tensor([[0.2635, 0.7365, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000]])
tensor([0.0086])
tensor([[0.6579, 0.3421, 0.0000, 0.0000, 0.0000, 0.0000, 0.0

In [26]:
1 / (speeds_two_hot.sum(dim=0) / speeds_two_hot.sum())

tensor([ 8.5494, 26.1585, 17.4698, 20.9403, 30.3160, 10.4518,  2.0598,  7.9585,
            inf,     inf])

In [2]:
dataset[0]["plant2_batch"]["target_speed"], dataset[0]["step_idx"]

(array([[0.]], dtype=float32), array([0.], dtype=float32))

In [2]:
speeds = []

for i in range(len(dataset)):
    speeds.append(dataset[i]["plant2_batch"]["target_speed"][0][0])

In [3]:
len(speeds)

5120

In [7]:
import numpy as np

speeds = np.array(speeds)

In [15]:
np.sum(speeds < 0.2) / len(speeds) * 100

np.float64(7.5)

In [13]:
import numpy as np
import matplotlib.pyplot as plt

speeds = np.array(speeds)

plt.hist(speeds, bins=10)
plt.show()


In [8]:
import numpy as np

def quantize_points_l1(v, k=10, max_iter=100, tol=1e-6):
    """
    Находит k опорных точек (квантизация) для 1D данных v,
    минимизируя суммарную абсолютную ошибку (L1).

    Параметры:
        v : np.ndarray
            Вектор скоростей (1D)
        k : int
            Количество точек (по умолчанию 8)
        max_iter : int
        tol : float

    Возвращает:
        centers : np.ndarray shape (k,)
    """
    v = np.asarray(v, dtype=float).ravel()
    if v.size == 0:
        raise ValueError("Empty input array")

    # Инициализация — квантили (хороший старт)
    centers = np.quantile(v, np.linspace(0, 1, k))

    for _ in range(max_iter):
        # Назначение к ближайшему центру (L1 и L2 дают одинаковое assignment в 1D)
        dist = np.abs(v[:, None] - centers[None, :])
        labels = np.argmin(dist, axis=1)

        new_centers = centers.copy()

        for j in range(k):
            cluster = v[labels == j]
            if cluster.size > 0:
                # КЛЮЧ: медиана → минимум L1 ошибки
                new_centers[j] = np.median(cluster)

        new_centers.sort()

        # проверка сходимости
        if np.max(np.abs(new_centers - centers)) < tol:
            break

        centers = new_centers

    return centers

In [10]:
quantize_points_l1(speeds, k=15)

array([ 0.02625312,  1.20466453,  2.36664581,  3.39765358,  4.27554822,
        4.92096663,  5.43035984,  5.84643793,  6.31233168,  6.82659435,
        7.36065888,  8.00865746,  9.04530287, 10.03672028, 11.18059444])

In [1]:
import os
import sys
import argparse
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch


# FILE_PATH = Path(__file__).resolve()
SDC_ROOT = Path("/home/jovyan/shares/SR006.nfs2/arbelyaev/sdc")
PDD_BENCH_DIR = SDC_ROOT / "pdd-bench"
METADRIVE_DIR = SDC_ROOT / "metadrive"
ARBELYAEV_ROOT = SDC_ROOT.parent
PLANT2_DIR = ARBELYAEV_ROOT / "plant2"
PLANT_PLAN_T_DIR = PLANT2_DIR / "PlanT"

for _p in (SDC_ROOT, METADRIVE_DIR, PDD_BENCH_DIR, PLANT_PLAN_T_DIR, PLANT2_DIR):
    _ps = str(_p)
    if _ps not in sys.path:
        sys.path.insert(0, _ps)


def _mock_carla_modules():
    import unittest.mock as _mock
    for mod_name in ("carla", "agents", "agents.navigation",
                     "agents.navigation.global_route_planner"):
        if mod_name not in sys.modules:
            sys.modules[mod_name] = _mock.MagicMock()

In [2]:
import torch
import os
import sys


def load_plant_model(checkpoint_path: str, plant_planT_path: str, device: str = "cpu"):
    import yaml
    _mock_carla_modules()

    model_yaml = os.path.join(plant_planT_path, "config", "model", "PlanT.yaml")
    if not os.path.isfile(model_yaml):
        raise FileNotFoundError(f"PlanT config not found: {model_yaml}")
    with open(model_yaml) as f:
        plnt = yaml.safe_load(f)

    class DictAsMember(dict):
        def __getattr__(self, name):
            value = self.get(name)
            if isinstance(value, dict) and not isinstance(value, DictAsMember):
                return DictAsMember(value)
            return value

    config_all = DictAsMember({"model": plnt})
    config_net = config_all.model.network

    if plant_planT_path not in sys.path:
        sys.path.insert(0, plant_planT_path)
    elif sys.path[0] != plant_planT_path:
        sys.path.remove(plant_planT_path)
        sys.path.insert(0, plant_planT_path)
    from model import HFLM  # type: ignore

    net = HFLM(config_net, config_all)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    sd = ckpt.get("state_dict", ckpt.get("model_state_dict", ckpt))
    if not isinstance(sd, dict):
        raise ValueError("Checkpoint contains no state_dict")
    if list(sd.keys())[0].startswith("model."):
        sd = {k.replace("model.", "", 1): v for k, v in sd.items()}
    net.load_state_dict(sd, strict=False)
    return net, config_all

net, config_all = load_plant_model(
    "/home/jovyan/shares/SR006.nfs2/arbelyaev/sdc/pdd-bench/outputs/plant2_supervised/plant2_supervised_2nd_final.pt", 
    "/home/jovyan/shares/SR006.nfs2/arbelyaev/plant2/PlanT", 
    device="cuda:3")

/home/jovyan/.mlspace/envs/plant2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
net.step_idx_emb.weight

Parameter containing:
tensor([[-0.0008],
        [-0.0132],
        [ 0.0101],
        [ 0.0227],
        [-0.0003],
        [ 0.0118],
        [ 0.0046],
        [ 0.0171],
        [ 0.0023],
        [-0.0141],
        [-0.0036],
        [ 0.0354],
        [-0.0339],
        [ 0.0182],
        [-0.0103],
        [-0.0079],
        [-0.0251],
        [ 0.0242],
        [ 0.0120],
        [-0.0069],
        [-0.0071],
        [-0.0194],
        [ 0.0504],
        [ 0.0272],
        [-0.0368],
        [ 0.0106],
        [-0.0044],
        [-0.0021],
        [ 0.0272],
        [-0.0194],
        [ 0.0476],
        [-0.0460],
        [ 0.0180],
        [ 0.0312],
        [ 0.0219],
        [-0.0468],
        [ 0.0177],
        [ 0.0135],
        [-0.0156],
        [ 0.0219],
        [ 0.0205],
        [-0.0065],
        [-0.0057],
        [-0.0099],
        [ 0.0415],
        [-0.0114],
        [ 0.0301],
        [-0.0049],
        [ 0.0319],
        [-0.0096],
        [-0.0080],
        [

In [12]:
net.step_idx_emb.bias

Parameter containing:
tensor([ 4.3837e-05, -7.1851e-05,  2.7569e-04,  1.1298e-04,  3.3138e-04,
         2.9360e-04,  3.1025e-04, -2.5583e-04,  2.4134e-04,  3.0547e-04,
         2.5910e-04, -4.0279e-05,  2.3580e-04,  1.5979e-04,  2.6246e-04,
        -2.3920e-04,  2.8597e-04,  2.9628e-04, -7.7432e-05, -8.5078e-05,
         2.0401e-04,  1.3743e-04,  2.2405e-04,  1.9734e-04,  3.4416e-04,
        -2.6057e-04, -2.1504e-04,  3.1035e-04, -4.8601e-05, -2.7542e-04,
         2.2419e-04,  1.7820e-04, -1.0536e-04, -2.9073e-04,  3.1169e-04,
         6.9981e-05,  7.8403e-05,  3.0018e-04,  6.7986e-05, -2.2327e-04,
         6.6224e-05, -4.6042e-05,  5.2931e-05,  2.0682e-04,  3.8321e-05,
         3.0803e-04,  2.7735e-04, -2.2039e-04, -2.7931e-04, -5.4134e-05,
         1.5366e-04, -1.7558e-04,  2.4539e-04,  2.5457e-04, -1.4918e-04,
         2.1446e-04,  1.4169e-04,  3.2268e-04,  2.7350e-04, -2.7399e-04,
        -1.3196e-04, -1.3338e-04, -1.4172e-04,  2.8458e-04,  2.7971e-04,
        -2.3395e-04,  5.7368e